# Demo

### Architecture of the data migration
- One GBB cluster with 2 nodes
- One gpdb cluster on GBB with 48 segments


In [14]:
import psycopg2
conn_gbb=psycopg2.connect(dbname= 'postgres', host='10.244.0.1', 
port= '5432', user= 'gpadmin', password= 'changeme')
cur_gbb = conn_gbb.cursor()

In [15]:
cur_gbb.execute("SELECT count(*) FROM gp_segment_configuration;")
rows = cur_gbb.fetchall()
import pprint
pprint.pprint(rows)

[(49L,)]


In [16]:
cur_gbb.close()
conn_gbb.close()

- One kubernetes 1.16.2 over the GBB cluster
- Then we provision GP for K8s with 2 segments

```
$ kubectl apply -f demo.yaml
```
   
```yaml
apiVersion: "greenplum.pivotal.io/v1"
kind: "GreenplumCluster"
metadata:
  name: my-greenplum
spec:
  masterAndStandby:
    hostBasedAuthentication: |
      # host   all   gpadmin   1.2.3.4/32   trust
      # host   all   gpuser    0.0.0.0/0   md5
    memory: "800Mi"
    cpu: "0.5"
    storageClassName: "local-storage"
    storage: 1G
    antiAffinity: "no"
    workerSelector: {}
  segments:
    primarySegmentCount: 1
    memory: "800Mi"
    cpu: "0.5"
    storageClassName: "local-storage"
    storage: 2G
    antiAffinity: "no"
    workerSelector: {}
```

In [19]:
conn_gp4k8s=psycopg2.connect(dbname= 'postgres', host='10.244.0.8', 
port= '5432', user= 'gpadmin', password= 'changeme')
cur_gp4k8s = conn_gp4k8s.cursor()

In [20]:
cur_gp4k8s.execute("SELECT count(*) FROM gp_segment_configuration;")
rows = cur_gp4k8s.fetchall()
import pprint
pprint.pprint(rows)

[(3L,)]


In [21]:
cur_gp4k8s.close()
conn_gp4k8s.close()

- We install jupyter and tensorflow in the kubernetes cluster using helm

```
$ helm install --name notebook stable/tensorflow-notebook
```


### Load Data
- We load GBB Cluster with storm data from: https://www1.ncdc.noaa.gov/pub/data/swdi/stormevents/csvfiles/legacy/

```sql
[gpadmin@sdw5 ~]$ psql
psql (9.4.24)
Type "help" for help.

gpadmin=# COPY ugc_areas FROM '/home/gpadmin/legacy_data/ugc_areas.csv' DELIMITER ',' CSV HEADER;

```

In [34]:
import numpy as np
import psycopg2
conn_gbb=psycopg2.connect(dbname= 'gpadmin', host='10.244.0.1', 
port= '5432', user= 'gpadmin', password= 'changeme')

cur_gbb = conn_gbb.cursor()
cur_gbb.execute("SELECT * FROM storms LIMIT 10;")

data = np.array(cur_gbb.fetchall())

cur_gbb.close()
conn_gbb.close()

data

array([['200902', '18', '1723', '200902', '18', '1725', '25944',
        '151499', 'ALABAMA', '1', '2009', 'February', 'Hail', 'C', '23',
        'CHOCTAW', 'MOB', '2/18/2009 17:23', 'CST-6', '2/18/2009 17:25',
        '0', '0', '0', '0', '0.00K', '0.00K', 'Emergency Manager', '1',
        None, None, None, None, None, None, None, None, None, None, '2',
        'NE', 'BUTLER', None, None, None, '32.1', '-88.2', None, None,
        'A strong cold front brought large hail and damaging winds to southwest Alabama.',
        None, '20090402', '2214', '20090502', '1153', '4/2/2009 22:14',
        '5/2/2009 11:53', None, None],
       ['201301', '1', '0', '201301', '31', '2359', '70304', '422432',
        'GEORGIA', '13', '2013', 'January', 'Drought', 'Z', '159',
        'BROOKS', 'TAE', '01/01/2013 00:00:00', 'EST-5',
        '01/31/2013 23:59:00', '0', '0', '0', '0', '', '',
        'Drought Monitor', '', '', '', '', '', '', '', '', '', '', '',
        None, '', '', None, '', '', '', '', ''

### Example how to use `tensorflow` 

for more details: https://www.tensorflow.org/tensorboard/get_started

In [35]:
import tensorflow as tf
import datetime

In [36]:
!rm -rf /output/training_logs/*

rm: cannot remove '/output/training_logs/': Device or resource busy


In [37]:
mnist = tf.keras.datasets.mnist

(x_train, y_train),(x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

def create_model():
  return tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
  ])

In [38]:
model = create_model()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

log_dir="/output/training_logs/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

model.fit(x=x_train, 
          y=y_train, 
          epochs=5, 
          validation_data=(x_test, y_test), 
          callbacks=[tensorboard_callback])

Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
Instructions for updating:
Apply a constraint manually following the optimizer update step.
Train on 60000 samples, validate on 10000 samples
Epoch 1/5
60000/60000 [==============================] - 4s 70us/sample - loss: 0.2194 - accuracy: 0.9343 - val_loss: 0.1109 - val_accuracy: 0.9684
Epoch 2/5
60000/60000 [==============================] - 4s 71us/sample - loss: 0.0967 - accuracy: 0.9701 - val_loss: 0.0872 - val_accuracy: 0.9750
Epoch 3/5
60000/60000 [==============================] - 4s 68us/sample - loss: 0.0687 - accuracy: 0.9782 - val_loss: 0.0702 - val_accuracy: 0.9783
Epoch 4/5
60000/60000 [==============================] - 4s 72us/sample - loss: 0.0541 - accuracy: 0.9825 - val_loss: 0.0818 - val_accuracy: 0.9751
Epoch 5/5
60000/60000 [==============================] - 4s 73us/sample - loss: 0.0434 - accuracy: 0.9859 - val_loss: 0.0739 - val_accuracy: 0.9788
